In [4]:
# 00. 데이터 불러오기
import pandas as pd

df_transcript_profile = pd.read_csv("transcript_profile.csv")
df_transcript_portfolio = pd.read_csv("transcript_portfolio.csv")
df_transcript_portfolio_profile = pd.read_csv("transcript_portfolio_profile.csv")

print(df_transcript_profile.shape)
print(df_transcript_portfolio.shape)
print(df_transcript_portfolio_profile.shape)

(306137, 11)
(306137, 15)
(306137, 20)


In [5]:
df_transcript_portfolio_profile.head()

,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,profile_missing
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,1
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,0
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26,1
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0


In [6]:
df = df_transcript_portfolio_profile.copy()

# 정렬 (필수)
df = df.sort_values(['customer_id', 'offer_id', 'time'])

result = []

for (cust, offer), group in df.groupby(['customer_id', 'offer_id']):
    received_times = group[group['event'] == 'offer received']['time'].tolist()
    completed_times = group[group['event'] == 'offer completed']['time'].tolist()

    for c in completed_times:
        # completed 이전에 received 있는지 체크
        valid_received = [r for r in received_times if r <= c]
        
        if not valid_received:
            result.append({
                'customer_id': cust,
                'offer_id': offer,
                'completed_time': c
            })

no_received_completed = pd.DataFrame(result)

In [7]:
df['event'].value_counts()

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33182
Name: count, dtype: int64

In [8]:
df[df['event'] == 'offer completed'].shape[0]

33182

In [9]:
print("completed 수:", len(df[df['event']=='offer completed']))
print("received 수:", len(df[df['event']=='offer received']))

completed 수: 33182
received 수: 76277


event 필터링

In [10]:
events = df[df['event'].isin(['offer received', 'offer viewed', 'offer completed'])].copy()

In [11]:
funnel = events.groupby(['customer_id', 'offer_id', 'event']).size().unstack(fill_value=0).reset_index()

In [12]:
funnel['received'] = funnel.get('offer received', 0)
funnel['viewed'] = funnel.get('offer viewed', 0)
funnel['completed'] = funnel.get('offer completed', 0)

전환고객 케이스분류 및 분석 converted_df

In [13]:
# 정렬
events = events.sort_values(['customer_id', 'offer_id', 'time'])

result = []

for (cust, offer), group in events.groupby(['customer_id', 'offer_id']):
    rec_times = group[group['event'] == 'offer received']['time'].tolist()
    view_times = group[group['event'] == 'offer viewed']['time'].tolist()
    comp_times = group[group['event'] == 'offer completed']['time'].tolist()
    
    # completed 기준으로 하나씩 판단
    for comp in comp_times:
        rec_before = any(r < comp for r in rec_times)
        view_before = any(v < comp for v in view_times)
        
        if rec_before and not view_before:
            case = 'received_no_view_completed'
        elif rec_before and view_before:
            case = 'received_view_completed'
        elif (not rec_before) and view_before:
            case = 'no_received_view_completed'
        else:
            case = 'other'
        
        result.append([cust, offer, comp, case])

converted_df = pd.DataFrame(result, columns=['customer_id', 'offer_id', 'time', 'case'])

In [14]:
converted_df['case'].value_counts()

case
received_view_completed       22313
received_no_view_completed     9199
other                          1670
Name: count, dtype: int64

In [15]:
#고객 정보 머지
converted_df = converted_df.merge(df[['customer_id', 'gender']], 
                        on='customer_id', how='left')

In [16]:
offer_map = df.drop_duplicates('offer_id').set_index('offer_id')['offer_type']

converted_df['offer_type'] = converted_df['offer_id'].map(offer_map)

In [17]:
converted_df['offer_type'].isna().sum()

np.int64(0)

In [18]:
df.groupby('offer_id')['offer_type'].nunique()

offer_id
0b1e1539f2cc45b7b9fa7c272da2e1d7    1
2298d6c36e964ae4a3e7e9706d1fb8c2    1
2906b810c7d4411798c6938adc9daaa5    1
3f207df678b143eea3cee63160fa8bed    1
4d5c57ea9a6940dd891ad53e9dbe8da0    1
5a8bc65990b245e5a138643cd4eb9837    1
9b98b8c7a33c4b65b9aebfe6a799e6d9    1
ae264e3637204a6fb9bb56bc8210ddfd    1
f19421c1d4aa40978ebb69ca19b0e20d    1
fafdcd668e3743c1bb461111dcafc2a4    1
Name: offer_type, dtype: int64

In [19]:
transactions = events[events['event'] == 'transaction'].copy()

In [20]:
purchase_summary = transactions.groupby('customer_id').agg(
    purchase_count=('amount', 'count'),
    avg_purchase=('amount', 'mean'),
    last_purchase=('time', 'max')
).reset_index()

In [21]:
converted_df = converted_df.merge(purchase_summary, on='customer_id', how='left')

In [22]:
max_time = transactions['time'].max()
converted_df['recency'] = max_time - converted_df['last_purchase']

In [55]:
print(converted_df['customer_id'].head())


0    0009655768c64bdeb2e877511632db8f
1    0009655768c64bdeb2e877511632db8f
2    0009655768c64bdeb2e877511632db8f
3    0009655768c64bdeb2e877511632db8f
4    0009655768c64bdeb2e877511632db8f
Name: customer_id, dtype: str


In [56]:
print(purchase_summary['customer_id'].head())

Series([], Name: customer_id, dtype: str)


비전환고객 non_converted_df

In [23]:
result = []

for (cust, offer), group in events.groupby(['customer_id', 'offer_id']):
    
    received_times = group[group['event'] == 'offer received']['time'].tolist()
    viewed_times = group[group['event'] == 'offer viewed']['time'].tolist()
    completed_times = group[group['event'] == 'offer completed']['time'].tolist()
    
    for r in received_times:
        
        # 다음 received 나오기 전까지만 보기 (핵심🔥)
        next_received = min([t for t in received_times if t > r], default=float('inf'))
        
        # 해당 구간 이벤트만 필터링
        viewed_in_window = [v for v in viewed_times if r < v < next_received]
        completed_in_window = [c for c in completed_times if r < c < next_received]
        
        viewed_flag = int(len(viewed_in_window) > 0)
        completed_flag = int(len(completed_in_window) > 0)
        
        # 비전환 케이스 정의
        if viewed_flag == 0 and completed_flag == 0:
            case = 'not_viewed'
        elif viewed_flag == 1 and completed_flag == 0:
            case = 'viewed_not_converted'
        else:
            case = 'other'  # 전환 or 기타
        
        result.append([cust, offer, r, viewed_flag, completed_flag, case])

In [24]:
non_converted_df = pd.DataFrame(result, columns=[
    'customer_id', 'offer_id', 'received_time',
    'viewed', 'completed', 'case'
])

In [25]:
non_converted = non_converted_df[non_converted_df['case'].isin(['not_viewed', 'viewed_not_converted'])]

In [26]:
non_converted['case'].value_counts()

case
viewed_not_converted    25064
not_viewed              20208
Name: count, dtype: int64

In [28]:
#고객정보 머지
non_converted_df = non_converted_df.merge(df[['customer_id', 'gender']], 
                        on='customer_id', how='left')

In [29]:
offer_map = df.drop_duplicates('offer_id').set_index('offer_id')['offer_type']

non_converted_df['offer_type'] = non_converted_df['offer_id'].map(offer_map)

In [30]:
non_converted_df['offer_type'].isna().sum()

np.int64(0)

In [31]:
df.groupby('offer_id')['offer_type'].nunique()

offer_id
0b1e1539f2cc45b7b9fa7c272da2e1d7    1
2298d6c36e964ae4a3e7e9706d1fb8c2    1
2906b810c7d4411798c6938adc9daaa5    1
3f207df678b143eea3cee63160fa8bed    1
4d5c57ea9a6940dd891ad53e9dbe8da0    1
5a8bc65990b245e5a138643cd4eb9837    1
9b98b8c7a33c4b65b9aebfe6a799e6d9    1
ae264e3637204a6fb9bb56bc8210ddfd    1
f19421c1d4aa40978ebb69ca19b0e20d    1
fafdcd668e3743c1bb461111dcafc2a4    1
Name: offer_type, dtype: int64

In [ ]:
#transactions = events[events['event'] == 'transaction'].copy()

In [ ]:
#purchase_summary = transactions.groupby('customer_id').agg(
#    purchase_count=('amount', 'count'),
#    avg_purchase=('amount', 'mean'),
#    last_purchase=('time', 'max')
#).reset_index()

In [32]:
non_converted_df = non_converted_df.merge(purchase_summary, on='customer_id', how='left')

In [33]:
max_time = transactions['time'].max()
non_converted_df['recency'] = max_time - non_converted_df['last_purchase']

received 기준 통합코드 case_df

In [34]:
result = []

for (cust, offer), group in events.groupby(['customer_id', 'offer_id']):
    
    received_times = group[group['event'] == 'offer received']['time'].tolist()
    viewed_times = group[group['event'] == 'offer viewed']['time'].tolist()
    completed_times = group[group['event'] == 'offer completed']['time'].tolist()
    
    for r in received_times:
        
        next_received = min([t for t in received_times if t > r], default=float('inf'))
        
        viewed_in_window = [v for v in viewed_times if r < v < next_received]
        completed_in_window = [c for c in completed_times if r < c < next_received]
        
        viewed_flag = int(len(viewed_in_window) > 0)
        completed_flag = int(len(completed_in_window) > 0)
        
        # 🔥 통합 케이스 정의
        if viewed_flag == 0 and completed_flag == 0:
            case = 'not_viewed'
        elif viewed_flag == 1 and completed_flag == 0:
            case = 'viewed_not_converted'
        elif viewed_flag == 1 and completed_flag == 1:
            case = 'received_view_completed'
        elif viewed_flag == 0 and completed_flag == 1:
            case = 'received_no_view_completed'
        else:
            case = 'other'
        
        result.append([cust, offer, r, viewed_flag, completed_flag, case])

case_df = pd.DataFrame(result, columns=[
    'customer_id', 'offer_id', 'received_time',
    'viewed', 'completed', 'case'
])

In [53]:
case_df['case'].value_counts()

case
received_view_completed       436410
viewed_not_converted          429438
not_viewed                    331862
received_no_view_completed    233051
Name: count, dtype: int64

In [35]:
#고객 정보 머지
case_df = case_df.merge(df[['customer_id', 'gender']], 
                        on='customer_id', how='left')

In [36]:
case_df['offer_type'] = case_df['offer_id'].map(offer_map)

In [37]:
case_df['offer_type'].isna().sum()

np.int64(0)

In [38]:
case_df = case_df.merge(purchase_summary, on='customer_id', how='left')

In [39]:
max_time = transactions['time'].max()
case_df['recency'] = max_time - case_df['last_purchase']

세그먼트별 분석 결과

1. 성별 분포 (convert,noncovert,case 순)

In [ ]:
#1-1.
converted_df.groupby('case')['gender'].value_counts(normalize=True)

case                        gender
other                       F         0.508436
                            M         0.470021
                            O         0.021543
received_no_view_completed  F         0.494277
                            M         0.492867
                            O         0.012857
received_view_completed     M         0.529646
                            F         0.453673
                            O         0.016681
Name: proportion, dtype: float64

In [47]:
#1-2.
non_converted_df.groupby('case')['gender'].value_counts(normalize=True)
case_df.groupby('case')['gender'].value_counts(normalize=True)

case                        gender
not_viewed                  M         0.626783
                            F         0.360736
                            O         0.012481
received_no_view_completed  M         0.513448
                            F         0.472391
                            O         0.014161
received_view_completed     M         0.522404
                            F         0.461531
                            O         0.016065
viewed_not_converted        M         0.624492
                            F         0.362424
                            O         0.013084
Name: proportion, dtype: float64

In [48]:
#1-3.
case_df.groupby('case')['gender'].value_counts(normalize=True)

case                        gender
not_viewed                  M         0.626783
                            F         0.360736
                            O         0.012481
received_no_view_completed  M         0.513448
                            F         0.472391
                            O         0.014161
received_view_completed     M         0.522404
                            F         0.461531
                            O         0.016065
viewed_not_converted        M         0.624492
                            F         0.362424
                            O         0.013084
Name: proportion, dtype: float64

2.offer_type 분포 (convert,noncovert,case 순)

In [46]:
#2-1.
converted_df.groupby(['case', 'offer_type']).size().unstack()

offer_type,bogo,discount
case,,
other,21342,16486
received_no_view_completed,87995,104249
received_view_completed,223348,266208


In [49]:
#2-2.
non_converted_df.groupby(['case', 'offer_type']).size().unstack()

offer_type,bogo,discount,informational
case,,,
not_viewed,101132.0,120038.0,110692.0
other,305942.0,363519.0,NaN
viewed_not_converted,171839.0,95543.0,162056.0


In [50]:
#2-3.
case_df.groupby(['case', 'offer_type']).size().unstack()

offer_type,bogo,discount,informational
case,,,
not_viewed,101132.0,120038.0,110692.0
received_no_view_completed,100220.0,132831.0,NaN
received_view_completed,205722.0,230688.0,NaN
viewed_not_converted,171839.0,95543.0,162056.0


3.구매횟수 (convert,noncovert,case 순)

In [52]:
#3-1.
converted_df.groupby('case')['purchase_count'].mean()

case
other                        NaN
received_no_view_completed   NaN
received_view_completed      NaN
Name: purchase_count, dtype: float64

In [ ]:
case_df.groupby('case')['avg_purchase'].mean()

case
not_viewed             NaN
other                  NaN
viewed_not_converted   NaN
Name: avg_purchase, dtype: float64